# 📧 Group Email Summarizer — Enron Dataset
**Full NLP pipeline: Load → Parse → Group → Summarise → Dashboard**

```
Enron emails.csv
      │
      ▼
EmailLoader   ─── parse RFC-2822 → group by subject thread
      │
      ▼
NLP Engine
  ├── sumy LSA       → extractive summary
  ├── KeyBERT        → key topics
  ├── VADER          → sentiment
  ├── spaCy NER      → owner detection
  └── Regex          → action items / follow-ups
      │
      ▼
pandas DataFrame  →  Excel Dashboard  +  Streamlit App
```

---
**No external API keys required. Runs entirely locally.**

## 0 — Install Dependencies

In [ ]:
!pip install sumy keybert vaderSentiment spacy streamlit plotly pandas openpyxl python-dateutil -q
!python -m spacy download en_core_web_sm -q
print('✓ All dependencies installed')

## 1 — Imports & Config

In [ ]:
import sys, re, json
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

# Add project root to path so utils imports work
sys.path.insert(0, str(Path('.').resolve()))

from utils.email_loader   import load_emails, group_into_threads, get_sample_threads
from utils.nlp_engine     import analyse_thread
from utils.excel_exporter import export_to_excel

# Config
USE_SAMPLE_DATA = True        # Set False to use real Enron CSV
ENRON_CSV_PATH  = 'data/emails.csv'
NROWS           = 80_000
TOP_N_THREADS   = 30

print('✓ Imports ready')

## 2 — Load Email Data

In [ ]:
if USE_SAMPLE_DATA or not Path(ENRON_CSV_PATH).exists():
    print('Using built-in sample threads (demo mode)')
    threads = get_sample_threads()
else:
    print(f'Loading from: {ENRON_CSV_PATH}')
    email_df = load_emails(csv_path=ENRON_CSV_PATH, nrows=NROWS)
    threads  = group_into_threads(email_df, min_emails=2, top_n=TOP_N_THREADS)

print(f'\n✓ {len(threads)} threads ready for analysis')
print('\nThread keys:')
for i, key in enumerate(list(threads.keys())[:5], 1):
    print(f'  {i}. {key}')
print('  ...')

## 3 — Preview a Sample Thread

In [ ]:
# Inspect the first thread's emails
first_key = list(threads.keys())[0]
first_emails = threads[first_key]

print(f'Thread: "{first_key}"  ({len(first_emails)} emails)\n')
print('=' * 60)
for i, e in enumerate(first_emails, 1):
    print(f'Email {i}:')
    print(f'  From   : {e["from_"]}')
    print(f'  Subject: {e["subject"]}')
    print(f'  Body   : {e["body"][:200]}…')
    print()

## 4 — Run NLP Analysis on All Threads

In [ ]:
results = []
total   = len(threads)

for i, (key, emails) in enumerate(threads.items(), 1):
    print(f'[{i:02d}/{total}] {key[:55]}')
    result = analyse_thread(key, emails)
    results.append(result)

df = pd.DataFrame(results)
print(f'\n✓ Done — {len(df)} threads analysed')

## 5 — Display Results Table

In [ ]:
SENT_COLORS = {
    'urgent'  : ('background-color:#fdecea; color:#c0392b', '🔴'),
    'negative': ('background-color:#fef0f0; color:#922b21', '🟠'),
    'positive': ('background-color:#eafaf1; color:#1e8449', '🟢'),
    'neutral' : ('background-color:#f8f9fa; color:#2c3e50', '⚪'),
}

def style_row(row):
    s = str(row.get('Sentiment','')).lower()
    css, _ = SENT_COLORS.get(s, ('', ''))
    return [css] * len(row)

display_cols = ['Email Thread', 'Key Topic', 'Action Items', 'Owner', 'Sentiment', 'Email Count']
df_display   = df[[c for c in display_cols if c in df.columns]].copy()

# Add emoji to sentiment
df_display['Sentiment'] = df_display['Sentiment'].apply(
    lambda s: f"{SENT_COLORS.get(s.lower(), ('','⚪'))[1]} {s.capitalize()}"
)

styled = (
    df_display.style
    .apply(style_row, axis=1)
    .set_caption('📧 Group Email Intelligence — Thread Analysis')
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color','#1f4e79'),('color','white'),
                  ('font-weight','bold'),('padding','10px 14px'),
                  ('font-size','11px'),('text-transform','uppercase'),
                  ('letter-spacing','.5px')]
    },{
        'selector': 'td',
        'props': [('padding','9px 14px'),('font-size','12px'),
                  ('vertical-align','top'),('max-width','300px'),
                  ('word-wrap','break-word')]
    },{
        'selector': 'caption',
        'props': [('font-size','14px'),('font-weight','bold'),
                  ('color','#1f4e79'),('margin-bottom','10px')]
    }])
    .set_properties(**{'text-align': 'left'})
)

display(styled)

## 6 — Deep Dive: Full Thread Analysis

In [ ]:
# Show full analysis for the first thread
sample = results[0]

html = f"""
<div style="font-family:Arial,sans-serif;max-width:800px">
  <div style="background:#1f4e79;color:white;padding:14px 20px;border-radius:8px 8px 0 0">
    <strong style="font-size:15px">📧 {sample['Email Thread']}</strong>
    <span style="float:right;font-size:11px;opacity:.7">{sample['Email Count']} emails · {sample['Latest Date']}</span>
  </div>
  <div style="border:1px solid #ddd;border-top:none;border-radius:0 0 8px 8px;padding:16px 20px">

    <div style="margin-bottom:14px">
      <div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.5px">Key Topic</div>
      <div style="font-size:13px;font-weight:bold;color:#1a5276">{sample['Key Topic']}</div>
    </div>

    <div style="margin-bottom:14px">
      <div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.5px">Summary</div>
      <div style="font-size:12px;color:#2c3e50;line-height:1.6">{sample['Summary']}</div>
    </div>

    <div style="margin-bottom:14px">
      <div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.5px">Action Items</div>
      {''.join(f"<div style='background:#f0f4ff;border-left:3px solid #5865f2;padding:6px 10px;margin:3px 0;font-size:11px'>▸ {a}</div>" for a in sample.get('Action List',[]) or sample['Action Items'].split(' | '))}
    </div>

    <div style="display:flex;gap:24px">
      <div>
        <div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.5px">Owner</div>
        <div style="font-size:13px;color:#6c3483;font-weight:bold">{sample['Owner']}</div>
      </div>
      <div>
        <div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.5px">Sentiment</div>
        <div style="font-size:13px">{sample['Sentiment'].capitalize()}</div>
      </div>
    </div>
  </div>
</div>
"""
display(HTML(html))

## 7 — Summary Statistics

In [ ]:
sent_counts  = df['Sentiment'].value_counts()
action_count = (df['Action Items'] != 'None identified').sum()

print('=' * 50)
print('  EMAIL THREAD SUMMARY STATISTICS')
print('=' * 50)
print(f'  Total threads analysed : {len(df)}')
print(f'  Total emails processed : {int(df["Email Count"].sum())}')
print(f'  Avg emails per thread  : {df["Email Count"].mean():.1f}')
print(f'  Threads with actions   : {int(action_count)}')
print()
print('  Sentiment Breakdown:')
for sent, cnt in sent_counts.items():
    bar = '█' * cnt
    print(f'    {sent.capitalize():<12} {cnt:>3}  {bar}')
print('=' * 50)

## 8 — Export to Excel Dashboard

In [ ]:
output_path = 'email_dashboard.xlsx'
export_to_excel(df, output_path=output_path)

# Also save CSV
df.to_csv('data/results.csv', index=False)
print('✓ CSV saved → data/results.csv')

display(HTML(f'<a href="{output_path}" style="color:#1a5276;font-size:14px">📥 Download {output_path}</a>'))